# Fraud detection – best CatBoost + Addition 2: re-anchoring non-SG amounts *and* transaction counts
This is `01_final_catboost.ipynb` (public PR-AUC **0.19586**) with **one added step**. The model, features, cross-validation and flag cut-off are unchanged. For non-Singapore countries, the test set's amount, 24h spend, **1h transaction count and 24h transaction count** are translated back onto each country's training scale before scoring. Addition 1 did the same for amounts only.

| | |
|---|---|
| **Base model** | CatBoost, 24 features, depth 4, lr 0.03 × 800, L2 10, seeds 0–2, 3 threads (unchanged) |
| **Added step** | Per-country quantile re-anchoring of test `transaction_amount`, `spend_last_24h`, `transactions_last_1h`, `transactions_last_24h`, non-SG countries only |
| **Retraining** | None needed: on training data the mapping is the identity, so retraining would give the same model |
| **Output** | `results/shift_reanchoring/submission_best_catboost_addition2.csv` |
| **Leaderboard so far** | plain 0.19586; Addition 1 (amounts only) 0.19544, no clear effect |
| **Safety check** | The unmapped path must still reproduce the 0.19586 file byte for byte (step 8) |

**Why (measured, label-free):**
- **Singapore is an unchanged control.** It is 63% of test rows, and its test amounts match training at every quantile (ratio 1.0).
- **Every other country's amounts are inflated in test:** most by 1.1–2.3×; AU and JP by 10–12× at the median (but ~2× at the 10th percentile and ~1.5× at the 90th); MY by 8× at the 75th percentile.
- **The best model's test top 2% is 89% non-SG.** On training data the same model's top 2% is 67% non-SG. The non-SG share of all rows only rose from 30% to 37%.
- **Transaction counts rose in non-SG markets too:** ≥ 4 per hour in MY 5.3% → 13.8%, GB 3.6% → 6.6%. After Addition 1, the biggest remaining gap in the test top 2% is bursts: 62% vs 41% in the same model's training top 2%.
- **So this version also re-anchors the counts.** The test bursts in those markets are judged against that market's training level of activity.

## 1. Setup and data
`THREADS = 3` is fixed on purpose. CatBoost splits the work into one block per thread, so **the thread count changes the fitted trees**: the same model with 3 threads vs all cores differs by up to 0.11 in individual probabilities. Fixing it makes the output reproducible on any machine, including Colab. It is the setting the 0.19586 submission was made with.

In [1]:
%pip install -q catboost scikit-learn pandas numpy
import os, sys, hashlib
import numpy as np
import pandas as pd

THREADS = 3                                        # fixed: CatBoost's result depends on the thread count (see above)
ROOT = ".." if os.path.isdir("../data") else "."
RESULTS = f"{ROOT}/results/shift_reanchoring"
os.makedirs(RESULTS, exist_ok=True)

# Read from the repo's data/ folder if it exists; otherwise open an upload button (Colab)
def load_csv(name):
    for path in [f"../data/{name}", f"data/{name}", name]:
        if os.path.exists(path):
            return pd.read_csv(path)
    from google.colab import files
    print(f"Upload: {name}")
    return pd.read_csv(list(files.upload().keys())[0])

df = load_csv("Track 2 Training Dataset.csv")
test_df = load_csv("Track 2 Testing Dataset.csv")
print("Train:", df.shape, " Test:", test_df.shape, f" fraud rate {df.fraud.mean():.2%}")

Note: you may need to restart the kernel to use updated packages.


Train: (20000, 12)  Test: (12000, 11)  fraud rate 1.76%


## 2. Cleaning (robust to messy new data)
The current files are clean, so this step changes nothing for them. It exists because the final ranking uses **unseen data** that may be messier:
- **Columns are selected by name;** a missing column is added as all-blank.
- **Numbers are parsed defensively:** text and infinities become blank.
- **Impossible values become blank** instead of being trusted: negative amounts, spend, counts or ages; hours outside 0–23; `new_device` other than 0/1.
- **The 1-hour count is capped at the 24-hour count,** since it can't logically exceed it.
- **Category spelling is normalised** to the training spelling (`" sg"` → `"SG"`); levels never seen in training become blank, and later `"Unknown"`.

Blanks are left as blanks (no zero-filling): CatBoost handles missing numbers natively, and categorical blanks get their own level.

In [2]:
target_column = "fraud"
CATS = ["merchant_category", "country", "transaction_channel"]
RAW_COLS = [c for c in df.columns if c not in ("id", target_column)]
NUMERIC_RAW = [c for c in RAW_COLS if c not in CATS]
NON_NEGATIVE = ["transaction_amount", "spend_last_24h", "transactions_last_1h", "transactions_last_24h", "account_age"]
# Training spelling of every category level, keyed by its trimmed lower-case form (e.g. " sg" -> "SG")
CAT_LEVELS = {c: {str(v).strip().lower(): v for v in df[c].dropna().unique()} for c in CATS}

def sanitise(d):
    d = d.reindex(columns=RAW_COLS).copy()                                 # select columns by name; missing ones become NaN
    for c in NUMERIC_RAW:
        d[c] = pd.to_numeric(d[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    for c in NON_NEGATIVE:
        d[c] = d[c].where(d[c] >= 0)                                       # negative -> NaN
    d["transaction_hour"] = d.transaction_hour.where(d.transaction_hour.between(0, 23))
    d["new_device"] = d.new_device.where(d.new_device.isin([0, 1]))
    over = d.transactions_last_1h > d.transactions_last_24h                # 1h count can't exceed the 24h count
    d.loc[over, "transactions_last_1h"] = d.loc[over, "transactions_last_24h"]
    for c in CATS:                                                         # normalise spelling; unseen levels -> NaN -> "Unknown"
        d[c] = d[c].astype("string").str.strip().str.lower().map(CAT_LEVELS[c]).astype(object)
    return d

## 3. Features
Each feature uses only that row's own values: no labels and no information from other rows, so there is no leakage. The 24 features fall into four groups.

| Group | Features | Why |
|---|---|---|
| **Raw columns (10)** | amount, hour, merchant, country, channel, 24h count, 24h spend, account age, new device, 1h count | The strongest signals live here: new device (7.5% fraud vs 1.2%), ≥ 5 transactions in an hour (23–30%), the top amount decile (5.6%), accounts under ~70 days old (4.6%). |
| **Smooth transforms** | `log_amt`, `log_spend`, `log_age`, `hour_sin`, `hour_cos` | Amounts and ages are heavily skewed, and logs give the trees evenly spread split points. Sine/cosine make 23:00 and 00:00 neighbours. |
| **Behaviour relative to the account itself** | `amt_to_avg_ticket` (this amount vs the account's average over 24h), `share_1h` (share of the last 24h's transactions that fell in the last hour) | Fraud is a transaction that is unusual *for that account*, which a raw amount can't express. |
| **Risk indicators from the EDA** | `night` (23:00–04:59), `burst` (≥ 4 in 1h), `high24` (≥ 11 in 24h), `young` (≤ 30 days), `risky_merchant` (luxury, cash transfer, electronics), `night_x_device`, `n_flags` (count of fired indicators) | They encode the EDA findings directly. Combinations are where fraud concentrates (pairs reach 38–40% fraud), and `n_flags` / `night_x_device` hand the shallow trees those combinations ready-made. |

**Categoricals** are passed to CatBoost as text; blanks become `"Unknown"`. CatBoost turns each level into a fraud-rate statistic computed only from *earlier* rows in a random order ("ordered target statistics"). That is its built-in protection against target leakage, so no manual target encoding is needed.

The test set gets exactly the same transformation and the training set's category lists.

In [3]:
def engineer(d):
    d = sanitise(d)
    d["log_amt"] = np.log1p(d.transaction_amount)
    d["log_spend"] = np.log1p(d.spend_last_24h)
    d["log_age"] = np.log1p(d.account_age)
    d["hour_sin"] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d["night"] = d.transaction_hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
    d["burst"] = (d.transactions_last_1h >= 4).astype(float)
    d["high24"] = (d.transactions_last_24h >= 11).astype(float)
    d["young"] = (d.account_age <= 30).astype(float)
    d["amt_to_avg_ticket"] = d.log_amt - np.log1p(d.spend_last_24h / d.transactions_last_24h.clip(lower=1))
    d["share_1h"] = d.transactions_last_1h / d.transactions_last_24h.clip(lower=1)
    d["risky_merchant"] = d.merchant_category.isin(["luxury", "cash_transfer", "electronics"]).astype(int)
    d["night_x_device"] = d.night * d.new_device
    d["n_flags"] = d[["burst", "high24", "young", "new_device", "night", "risky_merchant"]].fillna(0).sum(axis=1)
    for c in CATS:
        d[c] = d[c].fillna("Unknown").astype("category")
    return d

# The 24 model features, in the exact order used for the 0.19586 submission (CatBoost can depend on column order)
FEATURES = ["transaction_amount", "transaction_hour", "merchant_category", "country", "transaction_channel",
            "transactions_last_24h", "spend_last_24h", "account_age", "new_device", "transactions_last_1h",
            "log_amt", "log_spend", "log_age", "hour_sin", "hour_cos", "night", "burst", "high24", "young",
            "amt_to_avg_ticket", "share_1h", "risky_merchant", "night_x_device", "n_flags"]

X = engineer(df.drop(columns=["id", target_column]))[FEATURES]
y = df[target_column].values
X_test = engineer(test_df.drop(columns=["id"]))
for c in CATS:                                    # test categories use the training category lists
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=X[c].cat.categories)
X_test = X_test[FEATURES]

def as_catboost(d):                               # CatBoost takes categoricals as text
    return d.astype({c: str for c in CATS})

print(f"{len(FEATURES)} features; train {X.shape}, test {X_test.shape}")

24 features; train (20000, 24), test (12000, 24)


## 4. Model: CatBoost
**Why CatBoost:**
- **It held up best where it counts.** On the public leaderboard it beat every alternative we submitted: LightGBM 0.190, TabPFN 0.188, random forest blends, logistic blends (0.183), monotone-constrained CatBoost (~0.15).
- **It finds combinations of signals by itself,** and that's where fraud concentrates in this data.
- **Native categoricals with leakage-safe encoding,** and native handling of missing values.

**Settings and why:**

| Setting | Value | Why |
|---|---|---|
| `depth` | 4 | Small trees: each can combine up to 4 signals, enough for the fraud combinations, but too shallow to memorise individual rows among only 353 frauds. Depth 3 and 5 both scored lower in CV. |
| `learning_rate` × `iterations` | 0.03 × 800 | Many small steps generalise better than a few large ones. |
| `l2_leaf_reg` | 10 | Strong shrinkage of leaf values against noise from 353 frauds. 20–30 scored lower. |
| class weights | none | Weighting frauds up (`scale_pos_weight`, `SqrtBalanced`) lowered CV PR-AUC by 0.016–0.04. PR-AUC only needs a good *ranking*, and weighting distorts it. Recall comes from the cut-off instead (step 6). |
| seeds | 0, 1, 2, averaged | Averaging three fits reduces the randomness of a single fit. |
| `thread_count` | 3 | Reproducibility (see step 1). |

Searches over depth, L2, learning rate, feature subsampling, one-hot categoricals, coarser borders and Depthwise growth found nothing better than these settings.

In [4]:
from catboost import CatBoostClassifier

SEEDS = [0, 1, 2]

def make_catboost(seed):
    return CatBoostClassifier(iterations=800, depth=4, learning_rate=0.03, l2_leaf_reg=10, cat_features=CATS,
                              random_seed=seed, verbose=0, allow_writing_files=False, thread_count=THREADS)

def fit_predict(X_fit, y_fit, X_score, seeds):
    """Average fraud probability of one CatBoost per seed."""
    return np.mean([make_catboost(s).fit(as_catboost(X_fit), y_fit).predict_proba(as_catboost(X_score))[:, 1]
                    for s in seeds], axis=0)

## 5. Cross-validation
**5-fold stratified CV** (every fold keeps the 1.77% fraud rate). Each training row is scored once by a model that never saw it, giving out-of-fold predictions for all 20,000 rows. Two seeds per fold keep the run short. There's no timestamp and no account id, so a time-based or grouped split isn't possible.

**What CV can and can't tell us:**
- It measures how well the model ranks *training-like* data: PR-AUC ≈ 0.24 vs ≈ 0.018 for a random ranker.
- The test set is shifted. It has more new devices (9% → 15%), more bursts (2.3% → 4.4%), larger amounts and 3× more missing values, and those signals are fraud less often there. Every model we built scored ~0.045 lower on the leaderboard than in CV.
- So CV was used to choose *within* this design (settings, features), and the leaderboard to choose *between* designs.

In [5]:
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

CV_SEEDS = [0, 1]
oof = np.zeros(len(y))
for k, (tr, te) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(X, y)):
    oof[te] = fit_predict(X.iloc[tr], y[tr], X.iloc[te], CV_SEEDS)
    print(f"fold {k} done", flush=True)
print(f"Out-of-fold PR-AUC: {average_precision_score(y, oof):.4f}   (random ranker ~ {y.mean():.3f})")

fold 0 done


fold 1 done


fold 2 done


fold 3 done


fold 4 done


Out-of-fold PR-AUC: 0.2384   (random ranker ~ 0.018)


## 6. Cut-off for the fraud flags (F1 / Recall)
PR-AUC only uses the *ordering* of the scores, so no cut-off can change it. We checked this on the leaderboard: the same model scored the same with and without rescaling. F1 and Recall, however, need 0/1 flags.

- **Choosing the share:** for every share of rows flagged (0.2% to 6%, in 0.05% steps), build the confusion matrix on the out-of-fold scores and compute precision, recall and F1. Pick the share with the best F1, smoothed over 5 neighbouring shares so a one-off spike isn't chosen.
- **Mapping to 0.5:** most likely the organisers flag rows at 0.5, so scores are mapped monotonically into [0, 1] with that share above 0.5. The order is unchanged, so PR-AUC is unchanged.

In [6]:
def pick_share(score):
    rows = []
    for share in np.arange(0.002, 0.0601, 0.0005):
        flag = (score >= np.quantile(score, 1 - share)).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, flag).ravel()
        rows.append({"share": share, "TP": tp, "FP": fp, "FN": fn,
                     "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn), "F1": 2 * tp / (2 * tp + fp + fn)})
    cm = pd.DataFrame(rows)
    cm["F1_smooth"] = cm["F1"].rolling(5, center=True, min_periods=1).mean()   # don't pick a one-off noisy spike
    return cm.loc[cm["F1_smooth"].idxmax()]

best = pick_share(oof)
share = best["share"]
print(f"Flag the top {share:.2%} of rows: precision {best.precision:.3f}, recall {best.recall:.3f}, F1 {best.F1:.3f} "
      f"(TP {best.TP:.0f}, FP {best.FP:.0f}, FN {best.FN:.0f})")
pd.DataFrame([{"model": "catboost", "PR-AUC": average_precision_score(y, oof), "flag share": share, "precision": best.precision,
               "recall": best.recall, "F1": best.F1}]).round(4).to_csv(f"{RESULTS}/cv_metrics_best_catboost.csv", index=False)

Flag the top 1.65% of rows: precision 0.315, recall 0.295, F1 0.305 (TP 104, FP 226, FN 249)


## 7. Addition 2: re-anchor non-SG test amounts and transaction counts to their own market
**What it does.** For each country except Singapore, every test row's `transaction_amount` is replaced by the training amount at the **same percentile within that country**: the AU test 80th-percentile amount becomes the AU training 80th-percentile amount. The same is done for `spend_last_24h`, `transactions_last_1h` and `transactions_last_24h`. Then the 24 features are rebuilt from the corrected values, exactly as before.

**Counts are whole numbers,** so many test rows share the same count. Ties are broken at random with a fixed seed: a tiny random offset below 1, so rows with *different* counts never swap order. Each row then gets an actual training count (an `inverted_cdf` quantile, never an interpolated fraction). After the change, the 1h count is again capped at the 24h count by the cleaning step.

**What it keeps:**
- **The order of amounts within each country:** the largest AU test amount is still the largest AU amount, so the "large for this market" signal survives.
- **Singapore,** the unchanged control.
- **Blank values,** which stay blank.
- **Rows with a missing country,** left untouched (no market to anchor to).

**Why no retraining:**
- Applied to the training data, the mapping changes nothing, so the training set, features and model are exactly as before.
- Only the test *inputs* move back to the scale the model was trained on, like converting prices back into the training currency.
- It uses features only (training amounts and test amounts), never labels.

**What it assumes.** Within each non-SG country, the upward shift in amounts and activity is a market change, not more fraud. If the test set's extra bursts in MY/GB were mostly fraud, this step would hide them. It is a bigger change than Addition 1. The diagnostics in step 9 show how many top-ranked rows change, and the leaderboard decides.

In [7]:
REANCHOR_COLS = ["transaction_amount", "spend_last_24h"]
COUNT_COLS = ["transactions_last_1h", "transactions_last_24h"]   # whole numbers: ties broken at random
rng = np.random.default_rng(0)
ANCHOR = "SG"                                     # unchanged control market: left as it is
train_raw = df.drop(columns=["id", target_column])

def reanchor(test_raw):
    """Per non-SG country: map each test value to the training value at the same within-country percentile."""
    d = test_raw.copy()
    for c in sorted(train_raw.country.dropna().unique()):
        if c == ANCHOR:
            continue
        rows = (d.country == c).values
        for col in REANCHOR_COLS + COUNT_COLS:
            ref = train_raw.loc[train_raw.country == c, col].dropna().values
            vals = d.loc[rows, col]
            if len(ref) < 50 or vals.notna().sum() < 50:      # too few rows to estimate a distribution: leave as is
                continue
            if col in COUNT_COLS:                              # random tie-break inside equal counts (offset < 1 keeps order)
                pct = (vals + rng.random(len(vals)) * 0.999).rank(pct=True)
                mapped = np.quantile(ref, pct.fillna(0.5).clip(0, 1), method="inverted_cdf")   # an actual training count
            else:
                pct = vals.rank(pct=True)                      # within-country percentile in the test set (blanks stay blank)
                mapped = np.quantile(ref, pct.fillna(0.5).clip(0, 1))
            d.loc[rows, col] = np.where(vals.isna(), np.nan, mapped)
    return d

test_raw = test_df.drop(columns=["id"])
test_raw_reanchored = reanchor(test_raw)

def quantile_ratios(test_part):
    rows = []
    for c in sorted(train_raw.country.dropna().unique()):
        a = train_raw.loc[train_raw.country == c, "transaction_amount"]
        b = test_part.loc[test_part.country == c, "transaction_amount"]
        rows.append({"country": c, **{f"q{int(q*100)}": b.quantile(q) / a.quantile(q) for q in (0.1, 0.25, 0.5, 0.75, 0.9)}})
    return pd.DataFrame(rows).set_index("country")

print("Test / train amount ratio at each quantile, BEFORE re-anchoring:")
print(quantile_ratios(test_raw).round(2).to_string())
print("\nAFTER re-anchoring (non-SG should be ~1.0):")
print(quantile_ratios(test_raw_reanchored).round(2).to_string())
burst = lambda d: d.groupby("country").transactions_last_1h.apply(lambda s: (s >= 4).mean())
print("\nShare of rows with >= 4 transactions in the last hour, per country:")
print(pd.DataFrame({"train": burst(train_raw), "test before": burst(test_raw), "test after": burst(test_raw_reanchored)}).round(3).to_string())

def test_features(raw):
    Xt = engineer(raw)
    for c in CATS:
        Xt[c] = pd.Categorical(Xt[c].astype(str), categories=X[c].cat.categories)
    return Xt[FEATURES]

X_test_reanchored = test_features(test_raw_reanchored)

Test / train amount ratio at each quantile, BEFORE re-anchoring:
          q10   q25    q50   q75   q90
country                               
AU       2.24  3.72  10.62  7.33  1.74
GB       1.46  1.76   2.09  1.67  1.61
ID       1.06  1.31   1.12  1.33  1.53
JP       2.13  3.84  11.97  4.99  1.38
MY       1.63  1.51   2.15  8.14  5.01
PH       1.39  1.53   2.03  1.40  2.10
SG       1.02  1.01   0.98  0.87  1.03
TH       1.12  1.22   1.70  1.57  2.27
US       1.22  1.22   1.43  1.20  1.17
VN       1.53  1.29   1.82  1.64  2.17

AFTER re-anchoring (non-SG should be ~1.0):


          q10   q25   q50   q75   q90
country                              
AU       1.00  1.01  1.01  1.00  1.00
GB       1.02  1.01  1.00  1.01  1.00
ID       1.01  1.00  1.00  1.01  1.00
JP       1.00  1.00  1.00  1.01  1.00
MY       1.01  1.00  1.00  1.00  1.00
PH       1.01  1.01  1.00  1.00  1.00
SG       1.02  1.01  0.98  0.87  1.03
TH       1.00  1.01  1.01  1.00  1.00
US       1.01  1.01  1.00  1.00  1.00
VN       1.01  1.01  1.01  1.01  1.01

Share of rows with >= 4 transactions in the last hour, per country:
         train  test before  test after
country                                
AU       0.050        0.034       0.049
GB       0.036        0.066       0.036
ID       0.079        0.200       0.078
JP       0.037        0.047       0.037
MY       0.053        0.138       0.053
PH       0.034        0.067       0.035
SG       0.013        0.019       0.019
TH       0.031        0.059       0.031
US       0.039        0.040       0.040
VN       0.053        0.073       0

## 8. Final model and submissions
Train on all 20,000 rows (3 seeds) **once**, then score two versions of the test set:
1. **Original inputs.** Must reproduce the 0.19586 submission byte for byte. This proves nothing else changed.
2. **Re-anchored inputs (amounts and counts).** This is the new submission.

Both use the same flag share from step 6 (CV is unchanged, so the cut-off is too).

In [8]:
def to_submission_score(score, thr):
    """Monotone map to [0,1] that puts the flag cut-off at 0.5 (order, and so PR-AUC, unchanged)."""
    score = np.asarray(score, dtype=float)
    lo, hi = score.min(), score.max()
    below = 0.49 * (score - lo) / max(thr - lo, 1e-12)
    above = 0.51 + 0.49 * (score - thr) / max(hi - thr, 1e-12)
    return np.clip(np.where(score >= thr, above, below), 0, 1)

def make_submission(score):
    score = np.where(np.isfinite(score), score, np.nanmedian(score))       # defensive: a missing score gets the median
    sub = pd.DataFrame({"id": test_df["id"].values, "prediction": to_submission_score(score, np.quantile(score, 1 - share))})
    assert len(sub) == len(test_df) and sub["id"].is_unique and sub["prediction"].between(0, 1).all()
    for path in [f"{ROOT}/format/sample_submission.csv", "sample_submission.csv"]:
        if os.path.exists(path):
            assert (pd.read_csv(path)["id"].values == sub["id"].values).all(), "ids not in sample order"
    return sub

models = [make_catboost(s).fit(as_catboost(X), y) for s in SEEDS]
predict = lambda Xs: np.mean([m.predict_proba(as_catboost(Xs))[:, 1] for m in models], axis=0)
score_original = predict(X_test)
score_reanchored = predict(X_test_reanchored)

# Safety check: the original path must reproduce the 0.19586 submission exactly
EXPECTED_MD5 = "d666afc9c1454c2933b2aca8e21933fb"
md5 = hashlib.md5(make_submission(score_original).to_csv(index=False).encode()).hexdigest()
print("Original path MD5:", md5, "-> identical to the 0.19586 submission" if md5 == EXPECTED_MD5 else "-> DIFFERENT (check setup!)")
assert md5 == EXPECTED_MD5, "the base model no longer reproduces the 0.19586 submission"

sub = make_submission(score_reanchored)
OUT = f"{RESULTS}/submission_best_catboost_addition2.csv"
sub.to_csv(OUT, index=False)
print(f"{OUT}: {len(sub)} rows, flagged (>= 0.5): {(sub.prediction >= 0.5).sum()}")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(OUT)

Original path MD5: d666afc9c1454c2933b2aca8e21933fb -> identical to the 0.19586 submission
../results/shift_reanchoring/submission_best_catboost_addition2.csv: 12000 rows, flagged (>= 0.5): 198


## 9. Diagnostics and go / no-go before uploading
CV can't judge this step (on training data it changes nothing), so these label-free checks decide whether it is worth an upload:
- **Top-2% make-up:** the non-SG share should fall towards the ~67% the model puts in its top 2% on training data. The new-device and burst shares should **not** rise (otherwise the step just moves the problem elsewhere).
- **Size of the change:** the share of top-2% rows replaced. This step is bigger than Addition 1, so the limit is ~35%.

**Don't upload if** more than 35% of the top 2% is replaced, or if the new-device share rises by more than 3 points. The burst share is expected to *fall*.

**Reading the leaderboard result** (seed noise alone is about ±0.004):
- **≥ 0.200:** the activity shift in non-SG markets was misleading the model. Keep it.
- **0.192–0.200:** no clear effect.
- **< 0.192:** the extra non-SG activity carried real fraud signal. Drop it.

In [9]:
from scipy.stats import spearmanr
def describe(t):
    return {"non-SG": (t.country != "SG").mean(), "AU/JP/MY": t.country.isin(["AU", "JP", "MY"]).mean(),
            "new device": t.new_device.mean(), ">=4 tx in last 1h": (t.transactions_last_1h >= 4).mean(),
            "median amount (original)": t.transaction_amount.median()}
top_k = int(round(0.02 * len(test_df)))
top_orig, top_new = np.argsort(-score_original)[:top_k], np.argsort(-score_reanchored)[:top_k]
oof_top = oof >= np.quantile(oof, 0.98)
diag = pd.DataFrame({
    "training out-of-fold top 2% (reference)": describe(df[oof_top]),
    "test top 2%, original (0.19586)": describe(test_df.iloc[top_orig]),
    "test top 2%, re-anchored (addition 2)": describe(test_df.iloc[top_new]),
}).T
replaced = 1 - len(set(top_orig) & set(top_new)) / top_k
print(f"Rank correlation with the 0.19586 submission: {spearmanr(score_original, score_reanchored)[0]:.4f}")
print(f"Top-2% rows replaced: {replaced:.1%} ({round(replaced * top_k)} of {top_k})")
go = (replaced <= 0.35
      and diag.iloc[2]["new device"] <= diag.iloc[1]["new device"] + 0.03
)
print("Go / no-go:", "GO: the change is targeted and doesn't push new-device rows up" if go else "NO-GO: see the table")
diag.round(3).to_csv(f"{RESULTS}/diagnostics.csv")
diag.round(3)

Rank correlation with the 0.19586 submission: 0.9590
Top-2% rows replaced: 21.3% (51 of 240)
Go / no-go: GO: the change is targeted and doesn't push new-device rows up


,non-SG,AU/JP/MY,new device,>=4 tx in last 1h,median amount (original)
training out-of-fold top 2% (reference),0.675,0.210,0.737,0.408,525.660
"test top 2%, original (0.19586)",0.888,0.308,0.778,0.625,679.805
"test top 2%, re-anchored (addition 2)",0.779,0.208,0.808,0.558,477.790


## 9. What else was tried, and why this model was kept
All alternatives were judged on the public leaderboard (12,000 test rows, ~212 frauds). The same model moves ±0.004 just from the choice of seeds, so only bigger differences mean anything.

| Alternative | Public PR-AUC | Lesson |
|---|---|---|
| **This model** (CatBoost, 3 seeds) | **0.19586** | – |
| TabPFN + LightGBM + logistic blend | 0.19354 | More models did not help |
| CatBoost 1 seed / 10 seeds | 0.1925 / 0.1921 | Seed count is noise, not a lever |
| LightGBM | 0.1901 | – |
| Shift corrections (flag-excess correction + within-dataset context features) | 0.1893 | Reweighting signals for the test mix didn't help |
| Unlabelled test rows added as "not fraud" | 0.1878 | The test's extra flagged rows are not simply legitimate |
| TabPFN | 0.1875 | – |
| CatBoost + logistic 50/50 | 0.1832 | Logistic regression gives every flag a fixed bonus, which misfires on the shifted test set |
| Monotone-constrained CatBoost | ~0.15 | Forcing "more signal = more risk" everywhere amplified the misfiring flags |

Rejected on cross-validation before any upload:
- class weighting (−0.016 to −0.04);
- target / WoE encoding, missing-value indicators, splines and explicit interactions (all within noise);
- depth 3 / 5 and stronger regularisation;
- relative "context" features (−0.03);
- kNN / neural-net ensembles (−0.06).

**Model assumptions** (for the report):
1. **Rows are independent.** There is no account id or timestamp to model sequences.
2. **The training labels are correct.**
3. **Fraud signals in the test set point the same way as in training.** This is the assumption the train → test shift strains: the same flags are fraud less often in the test set, which is why every model scores ~0.045 lower there than in CV.

**Addition 1** (amounts only) scored 0.19544 on the leaderboard: no clear effect. **Addition 2** (this notebook) also re-anchors the transaction counts, the largest remaining gap between the test top 2% and the training top 2%.